### 🛠️ Initialize Notebook Variables

**Only modify entries under _USER CONFIGURATION_.**

In [ ]:
from pathlib import Path

import utils
from apimtypes import *
from console import print_error, print_info, print_ok, print_val, print_warning
from azure_resources import get_infra_rg_name, get_account_info

# ------------------------------
#    USER CONFIGURATION
# ------------------------------

rg_location = 'eastus2'
index       = 1
apim_sku    = APIM_SKU.BASICV2              # Options: 'BASICV2', 'STANDARDV2', 'PREMIUMV2'
deployment  = INFRASTRUCTURE.SIMPLE_APIM    # Options: see supported_infras below
api_prefix  = 'appid-cost-'                 # ENTER A PREFIX FOR THE APIS TO REDUCE COLLISION POTENTIAL WITH OTHER SAMPLES
tags        = ['costing', 'emit-metric', 'entra-appid']  # ENTER DESCRIPTIVE TAG(S)

# Sample data generation
generate_sample_load = True                 # Generate sample API calls to demonstrate cost tracking
sample_requests_per_caller = 50             # Base request count per simulated caller (multiplied by weight)

# Budget alerts
alert_threshold = 1000                      # Request count threshold per caller per hour
alert_email = 'alerts@contoso.com'          # Email for alert notifications (leave empty to skip)



# ------------------------------
#    SYSTEM CONFIGURATION
# ------------------------------

sample_folder    = 'costing-entra-appid'
rg_name          = get_infra_rg_name(deployment, index)
supported_infras = [INFRASTRUCTURE.AFD_APIM_PE, INFRASTRUCTURE.APIM_ACA, INFRASTRUCTURE.APPGW_APIM, INFRASTRUCTURE.APPGW_APIM_PE, INFRASTRUCTURE.SIMPLE_APIM]
nb_helper        = utils.NotebookHelper(sample_folder, rg_name, rg_location, deployment, supported_infras, True, index = index, apim_sku = apim_sku)

# Load the emit-metric policy XML
emit_metric_policy_path = utils.determine_policy_path('emit_metric_caller_id.xml', sample_folder)
emit_metric_policy_xml = Path(emit_metric_policy_path).read_text(encoding='utf-8')

# Define the API and its operations
api_path = 'appid-cost-demo'
cost_demo_get = GET_APIOperation2('get-status', 'Get Status', '/get', 'Get Status')

apis = [
    API(f'{api_prefix}cost-tracking-api', 'Cost Tracking by App ID', api_path,
        'API for demonstrating cost tracking by Entra ID application',
        policyXml = emit_metric_policy_xml,
        operations = [cost_demo_get], tags = tags, subscriptionRequired = True, serviceUrl = 'https://httpbin.org')
]

# Simulated caller app IDs (in a real scenario these would be actual Entra ID app registrations)
simulated_callers = [
    {'appid': 'a5846c0e-1111-4000-8000-000000000001', 'name': 'HR Service', 'request_weight': 1.0},
    {'appid': '9e6bfb3f-2222-4000-8000-000000000002', 'name': 'Finance Portal', 'request_weight': 2.5},
    {'appid': 'c3d2e1f0-3333-4000-8000-000000000003', 'name': 'Mobile Gateway', 'request_weight': 0.5},
    {'appid': 'b7a8c9d0-4444-4000-8000-000000000004', 'name': 'Engineering Tools', 'request_weight': 3.0}
]

# Get Azure account information
current_user, current_user_id, tenant_id, subscription_id = get_account_info()

if not subscription_id:
    print_error('Could not determine Azure subscription ID. Run: az login')
    raise SystemExit(1)

print_ok('Notebook initialized')

### 🚀 Deploy Infrastructure and APIs

Creates the bicep deployment into the previously-specified resource group. A bicep parameters, `params.json`, file will be created prior to execution.

In [ ]:
# Build the bicep parameters
bicep_parameters = {
    'location'  : {'value': rg_location},
    'index'     : {'value': index},
    'apis'      : {'value': [api.to_dict() for api in apis]}
}

# Deploy the sample
output = nb_helper.deploy_sample(bicep_parameters)

if output.success:
    # Extract deployment outputs
    apim_name                      = output.get('apimServiceName', 'APIM Service Name')
    apim_gateway_url               = output.get('apimResourceGatewayURL', 'APIM API Gateway URL')
    app_insights_name              = output.get('applicationInsightsName', 'Application Insights Name')
    app_insights_connection_string = output.get('applicationInsightsConnectionString', '')
    log_analytics_name             = output.get('logAnalyticsWorkspaceName', 'Log Analytics Workspace Name')
    workbook_name                  = output.get('workbookName', 'Workbook Name')
    workbook_id                    = output.get('workbookId', '')

    # Extract subscription key for the API
    api_outputs = output.getJson('apiOutputs', 'APIs')
    subscription_key = api_outputs[0]['subscriptionPrimaryKey'] if api_outputs else None

    print_ok('Deployment completed successfully')
else:
    print_error('Deployment failed!')
    raise SystemExit(1)

### 🚀 Generate Sample API Traffic

Generate sample API calls simulating different Entra ID applications (via crafted JWT `appid` claims) to demonstrate caller-based cost tracking.

This creates `caller-requests` custom metrics in Application Insights via the `emit-metric` policy.

> **Note:** In a production environment, callers would present real Entra ID tokens. Here we simulate them with minimal JWTs so that the `emit-metric` policy can extract the `appid` claim.

In [ ]:
import base64
import json
import time

from apimrequests import ApimRequests

if 'apim_gateway_url' not in locals():
    print_error('Please run the deployment cell first')
    raise SystemExit(1)

def make_fake_jwt(appid: str) -> str:
    """Create a minimal unsigned JWT with an appid claim for emit-metric extraction."""
    header = base64.urlsafe_b64encode(json.dumps({'alg': 'none', 'typ': 'JWT'}).encode()).rstrip(b'=').decode()
    payload = base64.urlsafe_b64encode(json.dumps({'appid': appid}).encode()).rstrip(b'=').decode()
    return f'{header}.{payload}.'

if generate_sample_load:
    print_info('Generating sample API traffic with simulated caller app IDs...')

    # Determine endpoints, URLs, etc. prior to test execution
    endpoint_url, request_headers = utils.get_endpoint(deployment, rg_name, apim_gateway_url)

    for caller in simulated_callers:
        caller_request_count = max(1, int(sample_requests_per_caller * caller.get('request_weight', 1.0)))
        fake_jwt = make_fake_jwt(caller['appid'])

        # Merge auth header with any infrastructure-specific headers
        auth_headers = dict(request_headers) if request_headers else {}
        auth_headers['Authorization'] = f'Bearer {fake_jwt}'

        reqs = ApimRequests(endpoint_url, subscription_key, auth_headers)
        reqs.multiGet(
            f'/{api_path}/get', caller_request_count,
            msg=f'Generating {caller_request_count} requests for {caller["name"]} ({caller["appid"][:12]}...)',
            printResponse=False, sleepMs=10
        )

    print()
    print_info('Note: Custom metrics typically take 5-10 minutes to appear in Application Insights')
else:
    print_info('Sample load generation skipped (generate_sample_load = False)')

### 🔍 Verify Metric Ingestion

Waits for `caller-requests` custom metrics to arrive in Application Insights (auto-retries for up to 10 minutes).

In [ ]:
import json
import tempfile
import time
from pathlib import Path

from azure_resources import run

if 'app_insights_name' not in locals():
    print_error('Please run the deployment cell first')
    raise SystemExit(1)

print_info('Waiting for caller-requests custom metrics to arrive in Application Insights...')
print_info('Metric ingestion typically takes 5-10 minutes after generating traffic')
print()

print_val('Application Insights', app_insights_name)

# Build the Application Insights resource ID for the ARM query endpoint
app_insights_resource_id = (
    f'/subscriptions/{subscription_id}'
    f'/resourceGroups/{rg_name}'
    f'/providers/Microsoft.Insights/components/{app_insights_name}'
)

# Load KQL from external file and wrap it in a JSON body
kql_path = utils.determine_policy_path('verify-metric-ingestion.kql', sample_folder)
kql_query = Path(kql_path).read_text(encoding='utf-8')

query_body = {'query': kql_query}

with tempfile.NamedTemporaryFile(mode='w', suffix='.json', delete=False) as f:
    json.dump(query_body, f)
    query_file_path = f.name

# Poll Application Insights until caller-requests metrics appear
max_wait_minutes = 10
poll_interval_seconds = 30
max_attempts = (max_wait_minutes * 60) // poll_interval_seconds
metrics_found = False

try:
    for attempt in range(1, max_attempts + 1):
        result = run(
            f'az rest --method POST '
            f'--url "https://management.azure.com{app_insights_resource_id}/api/query?api-version=2020-10-05-preview" '
            f'--body @{query_file_path} -o json',
            log_command=False
        )

        if not result.success:
            print_error(f'Query failed: {result.text[:300]}')
            break

        if result.json_data:
            tables = result.json_data.get('tables', [])
            if tables:
                rows = tables[0].get('rows', [])
                if rows and len(rows) > 0:
                    metric_count = float(rows[0][0])
                    if metric_count > 0:
                        print_ok(f'Found {int(metric_count)} caller-requests metric entries')
                        metrics_found = True
                        break

        elapsed = attempt * poll_interval_seconds
        remaining = (max_wait_minutes * 60) - elapsed
        print_info(f'  No metrics yet... retrying in {poll_interval_seconds}s ({remaining}s remaining)')
        time.sleep(poll_interval_seconds)
finally:
    Path(query_file_path).unlink(missing_ok=True)

if metrics_found:
    print_ok('Metric ingestion verified - workbook should now display data')
elif result.success:
    print_warning(f'Metrics did not appear within {max_wait_minutes} minutes')
    print_info('This can happen with newly deployed emit-metric policies. Tips:')
    print_info('  1. Wait a few more minutes and re-run this cell')
    print_info('  2. Verify the emit-metric policy is applied in Azure Portal')
    print_info('  3. Re-run the traffic generation cell to send more requests')

### 📊 Cost Analysis

#### Cost Allocation Model

| Component | Formula |
|---|---|
| **Base Cost** | Monthly platform cost for the APIM SKU |
| **Base Cost Share** | `Base Monthly Cost x (Caller Requests / Total Requests)` |
| **Total Allocated** | Proportional share of base cost per caller application |

The workbook in Application Insights visualises this model. Use the **Monthly Base Cost** parameter to adjust the base cost figure.

In [ ]:
base_monthly_cost = 150.00
print_val('Base monthly cost (default)', f'${base_monthly_cost:.2f}')

print()
print_info('The workbook uses the caller-requests custom metric from Application Insights.')
print_info('To adjust cost parameters, open the workbook and modify the parameters panel.')
print()
print_info('Key metric:')
print_val('customMetrics name', 'caller-requests')
print_val('Dimension', 'CallerId (Entra appid claim)')
print()
print_info('Sample KQL for ad-hoc analysis in Application Insights:')
print()
print('customMetrics')
print('| where name == "caller-requests"')
print('| extend CallerId = tostring(customDimensions.CallerId)')
print('| summarize RequestCount = sum(value) by CallerId')
print('| order by RequestCount desc')

### 🔔 Set Up Budget Alerts per Caller Application

Create Azure Monitor scheduled query alerts that fire when a caller application exceeds a configurable request threshold.

Each alert:
- Runs a KQL query every **5 minutes** against Application Insights
- Triggers when a caller exceeds the threshold in a **1-hour** rolling window
- Sends notifications via an **Action Group** (email)

> Adjust `alert_threshold` and `alert_email` in the initialization cell to match your requirements.

In [ ]:
import json
import tempfile
from pathlib import Path

from azure_resources import run

if not alert_email:
    print_warning('No alert_email configured - skipping budget alert setup')
    print_info('Set alert_email above to enable budget alerts per caller')
else:
    print_info('Setting up budget alerts per caller application...')

    # Get Application Insights resource ID
    ai_resource_id = (
        f'/subscriptions/{subscription_id}'
        f'/resourceGroups/{rg_name}'
        f'/providers/Microsoft.Insights/components/{app_insights_name}'
    )

    # Create an Action Group for alert notifications
    action_group_name = f'ag-appid-cost-alerts-{index}'
    print_info(f'Creating action group: {action_group_name}...')

    ag_result = run(
        f'az monitor action-group create '
        f'--resource-group {rg_name} '
        f'--name {action_group_name} '
        f'--short-name appidcost '
        f'--action email cost-alert-email {alert_email} '
        f'-o json',
        log_command=False
    )

    if ag_result.success:
        action_group_id = ag_result.json_data.get('id', '')
        print_ok(f'Action group created: {action_group_name}')
    else:
        print_error(f'Failed to create action group: {ag_result.text}')
        action_group_id = None

    if action_group_id:
        # Load the KQL template from an external file
        kql_path = utils.determine_policy_path('budget-alert-threshold.kql', sample_folder)
        kql_template = Path(kql_path).read_text(encoding='utf-8')

        print_info(f'Creating alerts for {len(simulated_callers)} caller applications (threshold: {alert_threshold} requests/hour)...')

        for caller in simulated_callers:
            safe_name = caller['name'].lower().replace(' ', '-')
            alert_name = f'apim-budget-{safe_name}-{index}'

            # Prepend KQL let bindings to parameterise the query
            kusto_query = f"let callerAppId = '{caller['appid']}';\nlet threshold = {alert_threshold};\n{kql_template}"

            alert_body = {
                'location': rg_location,
                'properties': {
                    'displayName': f'APIM Budget Alert: {caller["name"]}',
                    'description': f'Fires when {caller["name"]} ({caller["appid"]}) exceeds {alert_threshold} API requests per hour',
                    'severity': 2,
                    'enabled': True,
                    'evaluationFrequency': 'PT5M',
                    'windowSize': 'PT1H',
                    'scopes': [ai_resource_id],
                    'criteria': {
                        'allOf': [
                            {
                                'query': kusto_query,
                                'timeAggregation': 'Count',
                                'operator': 'GreaterThan',
                                'threshold': 0,
                                'failingPeriods': {
                                    'numberOfEvaluationPeriods': 1,
                                    'minFailingPeriodsToAlert': 1
                                }
                            }
                        ]
                    },
                    'actions': {
                        'actionGroups': [action_group_id]
                    }
                }
            }

            alert_id = (
                f'/subscriptions/{subscription_id}'
                f'/resourceGroups/{rg_name}'
                f'/providers/Microsoft.Insights/scheduledQueryRules/{alert_name}'
            )

            with tempfile.NamedTemporaryFile(mode='w', suffix='.json', delete=False) as f:
                json.dump(alert_body, f)
                alert_body_path = f.name

            try:
                result = run(
                    f'az rest --method PUT '
                    f'--uri https://management.azure.com{alert_id}?api-version=2023-03-15-preview '
                    f'--body @{alert_body_path}',
                    log_command=False
                )
            finally:
                Path(alert_body_path).unlink(missing_ok=True)

            if result.success:
                print_ok(f'  Alert created: {alert_name}')
            else:
                print_error(f'  Failed to create alert for {caller["name"]}: {result.text[:200]}')

        print()
        print_ok('Budget alerts configured!')
        print_val('Action Group', action_group_name)
        print_val('Alert Email', alert_email)
        print_val('Threshold', f'{alert_threshold} requests per hour per caller')
        print_val('Evaluation', 'Every 5 minutes, 1-hour rolling window')

### 🔗 Access Your Resources

Links to access the deployed resources in Azure Portal.

In [ ]:
print_info('Azure Portal Links:')
print()

base_url = 'https://portal.azure.com/#@/resource'
subscription_path = f'/subscriptions/{subscription_id}'
rg_path = f'{subscription_path}/resourceGroups/{rg_name}'

resources = [
    ('Application Insights', f'{rg_path}/providers/Microsoft.Insights/components/{app_insights_name}/overview'),
    ('Log Analytics Workspace', f'{rg_path}/providers/Microsoft.OperationalInsights/workspaces/{log_analytics_name}/Overview'),
    ('APIM Service', f'{rg_path}/providers/Microsoft.ApiManagement/service/{apim_name}/overview')
]

if 'workbook_id' in locals() and workbook_id:
    resources.append(('Azure Monitor Workbook', f'{base_url}{workbook_id}/workbook'))

for name, path in resources:
    print(f'{name}:')
    print(f'{base_url}{path}')
    print()

print_ok('Setup complete!')
print()
print_info('Cost Attribution is now tracking callers:')
print('  - emit-metric policy extracts appid/azp from JWT tokens')
print('  - caller-requests custom metric flows to Application Insights')
print('  - Azure Monitor Workbook visualises cost allocation per caller')
print()
print_info('Next steps:')
print('  - Open the Azure Monitor Workbook to view the cost dashboard')
print('  - Query Application Insights for detailed caller analysis')
print('  - Map App IDs to friendly names using the workbook parameter')